In [1]:
import os

%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\Chicken-Disease-Classification\\notebooks'

In [2]:
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\Chicken-Disease-Classification'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_classes: int
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    params_learning_rate: float
    
@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    checkpoint_dir: Path
    best_model_path: Path
    early_stopping_patience: int

In [4]:
from chicken_disease_classification.constants import *
from chicken_disease_classification.utils.common import read_yaml, create_directories

class ConfigManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_prepare_callbacks_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks_config
        create_directories([config.root_dir, config.checkpoint_dir])
        
        return PrepareCallbacksConfig(
            root_dir=Path(config.root_dir),
            checkpoint_dir=Path(config.checkpoint_dir),
            best_model_path=Path(config.best_model_path),
            early_stopping_patience=config.early_stopping_patience
        ) 
    
    def get_training_config(self) -> TrainingConfig:
        training_config = self.config.training_config
        create_directories([training_config.root_dir])
        
        prepare_base_model_config = self.config.prepare_base_model_config
        data_ingestion_config = self.config.data_ingestion_config
        
        training_config = TrainingConfig(
            root_dir=Path(training_config.root_dir),
            trained_model_path=Path(training_config.trained_model_path),
            updated_base_model_path=Path(prepare_base_model_config.updated_base_model_path),
            training_data=Path(data_ingestion_config.unzipped_data_dir),
            params_epochs=self.params.EPOCHS,
            params_batch_size=self.params.BATCH_SIZE,
            params_is_augmentation=self.params.AUGMENTATION,
            params_image_size=self.params.IMAGE_SIZE,
            params_classes=self.params.CLASSES,
            params_learning_rate=self.params.LEARNING_RATE,
        )
        
        return training_config

In [5]:
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path

from chicken_disease_classification.components.prepare_callbacks import PrepareCallbacks
from chicken_disease_classification.utils.dataloader import get_dataloaders
from chicken_disease_classification.utils.engine import train_one_epoch, validate

class Training:
    def __init__(self, config: TrainingConfig, callbacks_config: PrepareCallbacksConfig):
        self.config = config
        self.callbacks = PrepareCallbacks(config=callbacks_config)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    def get_base_model(self):
        self.model = models.vgg16()
        self.model.classifier[6] = nn.Linear(
            self.model.classifier[6].in_features,
            self.config.params_classes
        )
        
        weights = torch.load(self.config.updated_base_model_path, map_location=self.device)
        self.model.load_state_dict(weights)
        self.model = self.model.to(self.device)

    def get_dataloader(self):
        self.train_loader, self.val_loader = get_dataloaders(self.config)
        
    def train(self):
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=self.config.params_learning_rate
        )
        
        for epoch in range(self.config.params_epochs):
            train_loss = train_one_epoch(self.model, self.train_loader, criterion, optimizer, self.device)
            val_loss, accuracy = validate(self.model, self.val_loader, criterion, self.device)

            print(
                f"Epoch [{epoch+1}/{self.config.params_epochs}] "
                f"Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_loss:.4f} | "
                f"Accuracy: {accuracy:.4f}"
            )

            self.callbacks.save_checkpoint(self.model, optimizer, epoch, val_loss)
            self.callbacks.save_best_model(self.model, val_loss)

            if self.callbacks.check_early_stopping(val_loss):
                print(f"Early stopping at epoch {epoch+1}")
                break

        torch.save(self.model.state_dict(), self.config.trained_model_path)

In [6]:
import sys
from chicken_disease_classification.exception.exception import CustomException

try:
    config = ConfigManager()
    
    # Callbacks
    prepare_callbacks_config = config.get_prepare_callbacks_config()
    
    # Training
    training_config = config.get_training_config()
    training = Training(config=training_config, callbacks_config=prepare_callbacks_config)
    training.get_base_model()
    training.get_dataloader()
    training.train()

except Exception as e:
    raise CustomException(e, sys)

[2026-03-31 14:00:56,906: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-31 14:00:56,910: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-31 14:00:56,912: INFO: common: created directory at: artifacts]
[2026-03-31 14:00:56,913: INFO: common: created directory at: artifacts/prepare_callbacks]
[2026-03-31 14:00:56,915: INFO: common: created directory at: artifacts/prepare_callbacks/checkpoints]
[2026-03-31 14:00:56,916: INFO: common: created directory at: artifacts/training]
Epoch [1/1] Train Loss: 1.8158 | Val Loss: 0.3838 | Accuracy: 1.0000
